
# Contract Net Demo

Showcases the market-style coordinator where suppliers bid on tasks.


In [1]:

from agentic_codex import AgentBuilder, Context, ContractNetCoordinator
from agentic_codex.core.schemas import AgentStep, Message

GOAL = "Transcribe three audio clips with lowest estimated cost"
TASKS = ["clip-a", "clip-b", "clip-c"]

def auctioneer_step(ctx: Context) -> AgentStep:
    messages = [Message(role="auctioneer", content="Soliciting bids for audio clips")]
    return AgentStep(out_messages=messages, state_updates={"tasks": TASKS})

auctioneer = AgentBuilder(name="auctioneer", role="broker").with_step(auctioneer_step).build()

def make_bidder(name: str, base_cost: float):
    def bidder_step(ctx: Context) -> AgentStep:
        task = ctx.scratch.get("current_task", "unknown")
        score = base_cost + len(task) * 0.1
        content = f"{name} bids {score:.2f} for {task}"
        return AgentStep(
            out_messages=[Message(role="bidder", content=content)],
            state_updates={"score": score, "meta": {"task": task}},
        )

    return AgentBuilder(name=name, role="bidder").with_step(bidder_step).build()

bidders = {
    "fast-scribe": make_bidder("fast-scribe", 1.0),
    "economy-scribe": make_bidder("economy-scribe", 0.6),
}

contract_net = ContractNetCoordinator(auctioneer=auctioneer, bidders=bidders)
context = Context(goal=GOAL)
run = contract_net.run(goal=GOAL, inputs={})
[msg.content for msg in run.messages]


['Soliciting bids for audio clips',
 'fast-scribe bids 1.60 for clip-a',
 'economy-scribe bids 1.20 for clip-a',
 'fast-scribe bids 1.60 for clip-b',
 'economy-scribe bids 1.20 for clip-b',
 'fast-scribe bids 1.60 for clip-c',
 'economy-scribe bids 1.20 for clip-c']